# FIFA 2026 World Cup — Prediction System
## Notebook 5: Monte Carlo Tournament Simulator

### Where we are
Notebooks 3 and 4 built and validated our prediction engine:
- Poisson GLM baseline
- XGBoost Poisson
- Ensemble blend (70% GLM + 30% XGBoost)
- RPS calibration: 0.1670

### What this notebook does
Simulates the entire 2026 World Cup **10,000 times** using the
ensemble model to estimate probabilities for every possible outcome:
- Which teams qualify from each group
- Which teams reach the Round of 16, Quarter-finals, Semi-finals, Final
- Who wins the tournament

### How it works
For each simulation run:
1. Load all 104 WC26 fixtures
2. For each match — sample λ from Gamma distribution (Bayesian uncertainty)
3. Simulate scoreline from Poisson(λ)
4. Apply group stage tiebreakers (points → GD → goals → H2H → FIFA rank)
5. Qualify top 2 + best 8 third-place teams to Round of 32
6. Simulate knockout rounds until a champion is crowned

### Key design features
- **Bayesian λ sampling** — uncertainty scales with data availability
- **Full scoreline simulation** — not just W/D/L, goal difference matters
- **WC26 format encoded exactly** — 48 teams, 12 groups, correct tiebreakers
- **Heat stress applied** — venue heat index vs team tolerance for WC26 fixtures
- **10,000 runs** — enough for stable probability estimates

### Inputs
All pre-computed in Notebooks 3 and 4 — no retraining needed.

### Outputs
- `simulation_results.csv` — team probabilities at every stage
- `group_stage_results.csv` — average points, GD, goals per team

In [1]:
# Load all models and data 

import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
import sys
from pathlib import Path
from scipy.stats import poisson

# Paths
models_dir    = Path.home() / 'wc2026/models'
processed_dir = Path.home() / 'wc2026/data/processed'
raw_dir       = Path.home() / 'wc2026/data/raw'

# Load GLM models 
with open(models_dir / 'glm_home_v2.pkl', 'rb') as f:
    glm_home_v2 = pickle.load(f)
with open(models_dir / 'glm_away_v2.pkl', 'rb') as f:
    glm_away_v2 = pickle.load(f)

# Load XGBoost models 
xgb_home = xgb.Booster()
xgb_home.load_model(str(models_dir / 'xgb_home.json'))
xgb_away = xgb.Booster()
xgb_away.load_model(str(models_dir / 'xgb_away.json'))

# Load ensemble alpha and features 
with open(models_dir / 'ensemble_alpha.pkl', 'rb') as f:
    BEST_ALPHA = pickle.load(f)
with open(models_dir / 'features_v2.pkl', 'rb') as f:
    FEATURES_V2 = pickle.load(f)

# Load utility functions 
sys.path.insert(0, str(models_dir))
from utils import scoreline_matrix, match_probabilities

# Load WC26 fixtures 
fixtures  = pd.read_csv(processed_dir / 'fixtures_2026.csv', parse_dates=['date'])

# Load team data 
elo       = pd.read_csv(processed_dir / 'elo_ratings_2026.csv')
fifa      = pd.read_csv(processed_dir / 'fifa_rankings_2026.csv')
squad     = pd.read_csv(processed_dir / 'squad_values_2026.csv')
venue_heat = pd.read_csv(processed_dir / 'venue_heat_index.csv')

print("Models loaded:")
print(f"  GLM home/away")
print(f"  XGBoost home/away")
print(f"  Best alpha:       {BEST_ALPHA}")
print(f"  Features:         {FEATURES_V2}")

print(f"\nData loaded:")
print(f"  Fixtures:   {len(fixtures)} matches")
print(f"  Elo:        {len(elo)} teams")
print(f"  FIFA:       {len(fifa)} teams")
print(f"  Squad:      {len(squad)} teams")
print(f"  Venues:     {len(venue_heat)} venues")

Models loaded:
  GLM home/away
  XGBoost home/away
  Best alpha:       0.7
  Features:         ['elo_diff', 'squad_value_diff', 'home_form', 'away_form', 'home_advantage']

Data loaded:
  Fixtures:   104 matches
  Elo:        336 teams
  FIFA:       48 teams
  Squad:      48 teams
  Venues:     16 venues


In [2]:
# Check name consistency across all sources

fixture_teams = sorted([
    t for t in pd.concat([fixtures['home_team'], fixtures['away_team']]).unique()
                        if not str(t).startswith(('W','L','1','2','3'))
])

print(f"Total WC teams in fixtures: {len(fixture_teams)}")

# Check which teams are missing from each source
missing_elo = [t for t in fixture_teams if t not in elo['team'].values]
missing_fifa  = [t for t in fixture_teams if t not in fifa['team'].values]
missing_squad = [t for t in fixture_teams if t not in squad['team'].values]

print(f"\nMissing from Elo:   {missing_elo   if missing_elo   else 'None'}")
print(f"Missing from FIFA:  {missing_fifa  if missing_fifa  else 'None'}")
print(f"Missing from Squad: {missing_squad if missing_squad else 'None'}")

Total WC teams in fixtures: 48

Missing from Elo:   ['United States']
Missing from FIFA:  ['United States']
Missing from Squad: ['United States']


In [3]:
# Fix name mismatch 
# Fixtures use "United States" — elo, fifa, squad use "USA"
# Standardise all lookup tables to match fixtures

elo['team'] = elo['team'].replace({'USA':'United States'})
fifa['team'] = fifa['team'].replace({'USA':'United States'})
squad['team'] = squad['team'].replace({'USA':'United States'})

# Verify
missing_elo   = [t for t in fixture_teams if t not in elo['team'].values]
missing_fifa  = [t for t in fixture_teams if t not in fifa['team'].values]
missing_squad = [t for t in fixture_teams if t not in squad['team'].values]

print(f"Missing from Elo:   {missing_elo   if missing_elo   else 'None'}")
print(f"Missing from FIFA:  {missing_fifa  if missing_fifa  else 'None'}")
print(f"Missing from Squad: {missing_squad if missing_squad else 'None'}")

Missing from Elo:   None
Missing from FIFA:  None
Missing from Squad: None


In [4]:
# Build team feature lookup table 
# One row per team with all features needed for prediction.
# Built once and looked up instantly during simulation —
# avoids repeated merges inside the 10,000 run loop.

team_features = pd.DataFrame({'team':fixture_teams})

team_features = team_features.merge(
    elo[['team','elo_rating']],
    on = 'team',
    how = 'left'
).merge(
    fifa[['team','fifa_points','fifa_rank']],
    on = 'team',
    how = 'left'
).merge(
    squad[['team','squad_value_eur_m']],
    on = 'team',
    how = 'left'
)

# Set team as index for fast lookup during simulation
team_features = team_features.set_index('team')

print(f"Team lookup table shape: {team_features.shape}")
print(f"\nMissing values:")
print(team_features.isnull().sum())
print(f"\nSample:")
print(team_features.head(10).to_string())

Team lookup table shape: (48, 4)

Missing values:
elo_rating           0
fifa_points          0
fifa_rank            0
squad_value_eur_m    0
dtype: int64

Sample:
                      elo_rating  fifa_points  fifa_rank  squad_value_eur_m
team                                                                       
Algeria                   1826.5      1546.75         28                257
Argentina                 2073.6      1874.81          1                782
Australia                 1839.7      1557.30         27                 77
Austria                   1807.2      1570.20         24                245
Belgium                   1866.4      1734.71          9                548
Bosnia & Herzegovina      1624.3      1375.40         64                152
Brazil                    1956.1      1761.16          6                928
Canada                    1817.5      1522.44         30                199
Cape Verde                1634.0      1364.50         67                 55


### Match Prediction function


In [5]:
# Update predict_lambda with heat stress 
# Heat stress is applied as a post-prediction adjustment.
# Only affects WC26 fixtures — historical training untouched.
# A team playing in heat above their tolerance loses scoring ability.

import statsmodels.api as sm

# 
HEAT_PENALTY_COEF = 0.02  # 2% reduction in λ per heat unit above tolerance
                           # e.g. heat_index=8, tolerance=3 → 5 units → 10% penalty

def predict_lambda(home_team, away_team, home_advantage=0,
                   home_form=0.5, away_form=0.5,
                   venue_heat=None,
                   home_heat_tolerance=None,
                   away_heat_tolerance=None):

    home = team_features.loc[home_team]
    away = team_features.loc[away_team]

    elo_diff         = home['elo_rating']        - away['elo_rating']
    squad_value_diff = home['squad_value_eur_m'] - away['squad_value_eur_m']

    features = pd.DataFrame([{
        'elo_diff':          elo_diff,
        'squad_value_diff':  squad_value_diff,
        'home_form':         home_form,
        'away_form':         away_form,
        'home_advantage':    home_advantage
    }])

    # GLM prediction
    X      = sm.add_constant(features[FEATURES_V2], has_constant='add')
    lh_glm = float(glm_home_v2.predict(X).iloc[0])
    la_glm = float(glm_away_v2.predict(X).iloc[0])

    # XGBoost prediction
    dmat   = xgb.DMatrix(features[FEATURES_V2])
    lh_xgb = float(xgb_home.predict(dmat)[0])
    la_xgb = float(xgb_away.predict(dmat)[0])

    # Ensemble blend
    lh = BEST_ALPHA * lh_glm + (1 - BEST_ALPHA) * lh_xgb
    la = BEST_ALPHA * la_glm + (1 - BEST_ALPHA) * la_xgb

    # Apply heat stress if venue data provided
    if venue_heat is not None:
        home_heat_stress = max(0, venue_heat - home_heat_tolerance)
        away_heat_stress = max(0, venue_heat - away_heat_tolerance)

        lh = lh * (1 - HEAT_PENALTY_COEF * home_heat_stress)
        la = la * (1 - HEAT_PENALTY_COEF * away_heat_stress)

    # Asymmetric cap
    lh = min(lh, 6.0)
    la = min(la, 5.0)

    return lh, la


In [6]:
# Cache predict_lambda results 
# Same matchup in knockout rounds returns instantly from cache.
# Cache is cleared between runs automatically via lru_cache.

from functools import lru_cache

@lru_cache(maxsize=None)
def predict_lambda_cached(home_team, away_team, home_advantage=0,
                          home_form=0.5, away_form=0.5):
    """
    Cached version of predict_lambda for knockout rounds.
    Same matchup never computed twice across all 10,000 runs.
    """
    return predict_lambda(home_team, away_team,
                          home_advantage = home_advantage,
                          home_form      = home_form,
                          away_form      = away_form)

### Precompute λ for all 104 fixtures 

In [7]:
# Precompute λ for all 104 fixtures 
# predict_lambda is the bottleneck — GLM + XGBoost called 104 times per run
# Instead compute λ once for every fixture and store it
# During simulation just sample from Gamma(λ) — much faster

hosts = ['United States', 'Mexico', 'Canada']

fixture_lambdas = []

for _, row in fixtures.iterrows():
    home = row['home_team']
    away = row['away_team']

    # Skip placeholder teams in knockout fixtures
    if any(str(t).startswith(('W','L','1','2','3'))
           for t in [home, away]):
        fixture_lambdas.append({'lh': None, 'la': None})
        continue

    home_adv = 1 if (home in hosts and row['stage'] == 'Group') else 0

    lh, la = predict_lambda(
        home, away,
        home_advantage      = home_adv,
        venue_heat          = row.get('heat_index_scaled', None),
        home_heat_tolerance = row.get('home_heat_tolerance', None),
        away_heat_tolerance = row.get('away_heat_tolerance', None)
    )

    fixture_lambdas.append({
        'home':      home,
        'away':      away,
        'stage':     row['stage'],
        'group':     row.get('group', None),
        'lh':        lh,
        'la':        la
    })

lambda_df = pd.DataFrame(fixture_lambdas)

print(f"Precomputed λ for {lambda_df['lh'].notna().sum()} group stage fixtures")
print(lambda_df[lambda_df['lh'].notna()][
    ['home','away','stage','lh','la']
].head(10).to_string(index=False))

Precomputed λ for 72 group stage fixtures
          home                 away stage       lh       la
        Mexico         South Africa Group 2.495078 0.574295
   South Korea       Czech Republic Group 1.742705 1.104964
Czech Republic         South Africa Group 1.625273 1.055258
        Mexico          South Korea Group 2.018644 0.820881
Czech Republic               Mexico Group 1.095870 1.641288
  South Africa          South Korea Group 1.068538 1.667532
        Canada Bosnia & Herzegovina Group 2.337718 0.641701
         Qatar          Switzerland Group 0.912096 1.993484
   Switzerland Bosnia & Herzegovina Group 2.187221 0.850482
        Canada                Qatar Group 2.496474 0.582977


## Bayesian Lambda Sampling

### Why sample instead of using a fixed lambda?
`predict_lambda` returns a single point estimate — Spain always gets
λ=2.3, Cape Verde always gets λ=0.6. The upset can happen in the
scoreline matrix but the expected goals never vary. This makes the
model overconfident.

Bayesian sampling adds real day-to-day uncertainty — before each
simulation run we sample λ from a Gamma distribution:  

On a day when Saudi Arabia (ranked 53) beats Argentina (ranked 3)  from
the high end simultaneously — you get the shock result.

### Why Gamma distribution?
- Always positive — goals can never be negative
- Flexible shape — can be narrow or wide
- Natural conjugate prior for Poisson — mathematically consistent

### The uncertainty coefficient
Controls the width of the Gamma distribution:

| Value | Effect |
|---|---|
| 0.1 | Rare upsets — model too confident |
| 0.35 | Realistic upset rate (~10-15% for big mismatches) |
| 0.6 | Too many upsets — model unreliable |

We start at 0.35 and can tune later against historical upset rates.

### Knockout matches
No draws allowed in knockout rounds. If scores are level after
90 minutes we simulate a penalty shootout — 50/50 with slight
home advantage.

## simulate_match and simulate_group

### simulate_match
Takes two teams and match context, returns a simulated scoreline.

**Steps:**
1. Call `predict_lambda` to get ensemble λ point estimates
2. Sample λ from Gamma distribution (Bayesian uncertainty)
3. Simulate goals from Poisson(λ_sample)
4. If knockout and draw — simulate penalty shootout (50/50)

**Parameters:**
- `home_advantage` — 1 for USA/Mexico/Canada in group stage
- `venue_heat` — scaled heat index for the venue (1-10)
- `home/away_heat_tolerance` — team's climate adaptation (1-10)
- `uncertainty_coef` — controls upset frequency (default 0.35)
- `knockout` — True for Round of 32 onwards, no draws allowed

### simulate_group
Simulates all 6 matches in a 4-team group using fixture data.
Looks up venue heat directly from the fixtures table for each match
and passes it through to `simulate_match` automatically.

**What it tracks:**
- Points, wins, draws, losses per team
- Goals for, goals against, goal difference
- Head to head points and GD for tiebreakers

**Returns:**
- Standings table (Pos, Team, MP, W, D, L, GF, GA, GD, Pts)
- Full list of match results

### Heat stress chain
```
fixtures (venue heat data)
      ↓
simulate_group (looks up heat per match)
      ↓
simulate_match (passes heat through)
      ↓
predict_lambda (applies heat penalty to λ)
```

In [8]:
# simulate_match and simulate_group using precomputed λ 
# predict_lambda is no longer called during simulation.
# λ values are looked up from lambda_df — just Gamma sampling + Poisson.
# This makes each tournament run ~50x faster.

UNCERTAINTY_COEF = 0.35

def simulate_match_fast(lh, la, knockout=False):
    """
    Simulates a match from precomputed λ values.
    No predict_lambda call — just Bayesian sampling + Poisson.
    """
    lh_sample = np.random.gamma(lh / UNCERTAINTY_COEF, UNCERTAINTY_COEF)
    la_sample = np.random.gamma(la / UNCERTAINTY_COEF, UNCERTAINTY_COEF)

    home_goals = np.random.poisson(lh_sample)
    away_goals = np.random.poisson(la_sample)

    if knockout and home_goals == away_goals:
        if np.random.random() < 0.5:
            home_goals += 1
        else:
            away_goals += 1

    return home_goals, away_goals


def simulate_group_fast(group_name, lambda_df):
    """
    Simulates a group using precomputed λ values.
    """
    group_fix = lambda_df[lambda_df['group'] == group_name].copy()

    teams = pd.concat([
        group_fix['home'], group_fix['away']
    ]).unique().tolist()

    standings = {team: {
        'points': 0, 'gd': 0, 'gf': 0, 'ga': 0,
        'w': 0, 'd': 0, 'l': 0
    } for team in teams}

    h2h = {team: {'points': 0, 'gd': 0} for team in teams}
    matches = []

    for _, row in group_fix.iterrows():
        home = row['home']
        away = row['away']
        lh   = row['lh']
        la   = row['la']

        hg, ag = simulate_match_fast(lh, la)
        matches.append((home, away, hg, ag))

        standings[home]['gf'] += hg
        standings[home]['ga'] += ag
        standings[home]['gd'] += hg - ag
        standings[away]['gf'] += ag
        standings[away]['ga'] += hg
        standings[away]['gd'] += ag - hg

        if hg > ag:
            standings[home]['points'] += 3
            standings[home]['w'] += 1
            standings[away]['l'] += 1
            h2h[home]['points'] += 3
        elif hg == ag:
            standings[home]['points'] += 1
            standings[away]['points'] += 1
            standings[home]['d'] += 1
            standings[away]['d'] += 1
            h2h[home]['points'] += 1
            h2h[away]['points'] += 1
        else:
            standings[away]['points'] += 3
            standings[away]['w'] += 1
            standings[home]['l'] += 1
            h2h[away]['points'] += 3

        h2h[home]['gd'] += hg - ag
        h2h[away]['gd'] += ag - hg

    def sort_key(team):
        s = standings[team]
        return (
            -s['points'],
            -s['gd'],
            -s['gf'],
            -h2h[team]['points'],
            -h2h[team]['gd'],
            team_features.loc[team, 'fifa_rank']
        )

    ranked = sorted(teams, key=sort_key)

    result = []
    for pos, team in enumerate(ranked, 1):
        s = standings[team]
        result.append({
            'Pos':  pos,
            'Team': team,
            'MP':   3,
            'W':    s['w'],
            'D':    s['d'],
            'L':    s['l'],
            'GF':   s['gf'],
            'GA':   s['ga'],
            'GD':   s['gd'],
            'Pts':  s['points']
        })

    return pd.DataFrame(result), matches


def simulate_group_stage_fast(lambda_df):
    """
    Simulates all 12 groups using precomputed λ values.
    """
    group_names = sorted(lambda_df['group'].dropna().unique())

    all_standings    = {}
    group_winners    = []
    group_runners_up = []
    third_place      = []

    for group in group_names:
        standings, matches = simulate_group_fast(group, lambda_df)

        all_standings[group] = standings
        group_winners.append(standings.iloc[0]['Team'])
        group_runners_up.append(standings.iloc[1]['Team'])

        third = standings.iloc[2]
        third_place.append({
            'team':   third['Team'],
            'group':  group,
            'points': third['Pts'],
            'gd':     third['GD'],
            'gf':     third['GF']
        })

    best_third = rank_third_place(third_place, verbose=False)

    return {
        'all_standings':    all_standings,
        'group_winners':    group_winners,
        'group_runners_up': group_runners_up,
        'third_place':      third_place,
        'best_third':       best_third
    }


print("Fast simulation functions defined:")
print("  simulate_match_fast")
print("  simulate_group_fast")
print("  simulate_group_stage_fast")

Fast simulation functions defined:
  simulate_match_fast
  simulate_group_fast
  simulate_group_stage_fast


In [11]:
# Check heat tolernace in group stage
print(fixtures[['home_team', 'away_team', 'venue',
                'heat_index_scaled', 'home_heat_tolerance',
                'away_heat_tolerance']].head(10).to_string(index=False))

     home_team            away_team                                venue  heat_index_scaled  home_heat_tolerance  away_heat_tolerance
        Mexico         South Africa                          Mexico City                1.6                  7.0                  7.0
   South Korea       Czech Republic                Guadalajara (Zapopan)                2.6                  5.0                  2.0
Czech Republic         South Africa                              Atlanta                6.5                  2.0                  7.0
        Mexico          South Korea                Guadalajara (Zapopan)                2.6                  7.0                  5.0
Czech Republic               Mexico                          Mexico City                1.6                  2.0                  7.0
  South Africa          South Korea                Monterrey (Guadalupe)                6.8                  7.0                  5.0
        Canada Bosnia & Herzegovina                           

## Rank third place..

In [12]:
# rank_third_place 

def rank_third_place(third_place_teams, verbose=False):
    """
    Ranks all 12 third place teams and returns the best 8.
    third_place_teams — list of dicts with team stats from simulate_group
    """
    ranked = sorted(third_place_teams, key=lambda x: (
        -x['points'],
        -x['gd'],
        -x['gf'],
        team_features.loc[x['team'], 'fifa_rank']
    ))

    if verbose:
        print("All 12 third place teams ranked:")
        print(f"{'Pos':<5} {'Team':<25} {'Pts':<6} {'GD':<6} {'GF':<6} {'Qualified'}")
        print("-" * 60)
        for i, t in enumerate(ranked, 1):
            qualified = "YES" if i <= 8 else "no"
            print(f"{i:<5} {t['team']:<25} {t['points']:<6} {t['gd']:<6} {t['gf']:<6} {qualified}")
    return [t['team'] for t in ranked[:8]]

## Run_tournament — Full Single Simulation

### What it does
Runs one complete WC26 tournament from group stage to Final.
Every call produces a different result due to Bayesian λ sampling.

### Steps in one tournament run

**1. Group stage**
- Simulate all 12 groups (72 matches)
- Rank teams — 1st, 2nd, 3rd, 4th in each group
- Rank 12 third place teams — best 8 qualify
- 32 teams advance to Round of 32

**2. Round of 32 (32 matches)**
- Fixed bracket — group position determines the path
- No draws — penalty shootout if level
- 16 winners advance

**3. Round of 16 (16 matches)**
- 8 winners advance to Quarter-finals

**4. Quarter-finals (8 matches)**
- 4 winners advance to Semi-finals

**5. Semi-finals (4 matches)**
- 2 winners advance to Final
- 2 losers play Third place match

**6. Final (1 match)**
- Champion crowned

### Output
A dictionary tracking every team's furthest stage reached:
{

'Argentina': 'Winner',

'France':    'Final',

'Brazil':    'Semi-final',

'England':   'Quarter-final',

...

}

This is what gets counted across 10,000 runs to produce probabilities.

In [18]:
# run_tournament using fast functions 

def run_tournament_fast(lambda_df):
    """
    Simulates one complete WC26 tournament using precomputed λ.
    """
    team_stages = {team: 'Group Stage' for team in fixture_teams}

    # Group stage
    gs = simulate_group_stage_fast(lambda_df)

    for team in gs['group_winners']:
        team_stages[team] = 'Round of 32'
    for team in gs['group_runners_up']:
        team_stages[team] = 'Round of 32'
    for team in gs['best_third']:
        team_stages[team] = 'Round of 32'

    # Round of 32 bracket
    winners    = gs['group_winners']
    runners_up = gs['group_runners_up']
    best_third = gs['best_third']

    r32_matches = []
    for i in range(8):
        r32_matches.append((winners[i], best_third[i]))
    for i in range(4):
        r32_matches.append((winners[8 + i], runners_up[i]))
    for i in range(4):
        r32_matches.append((runners_up[4 + i], runners_up[8 + i]))

    # Knockout rounds — use simulate_match_fast with average λ
    def play_knockout_round(matches):
        winners_list = []
        for home, away in matches:
            lh, la = predict_lambda_cached(home, away)  # cached
            winner_goals, loser_goals = simulate_match_fast(lh, la, knockout=True)
            winner = home if winner_goals > loser_goals else away
            winners_list.append(winner)
        return winners_list
    
    # Round of 32
    r32_winners = play_knockout_round(r32_matches)
    for team in r32_winners:
        team_stages[team] = 'Round of 16'
        # losers stay at 'Round of 32'
    
    # Round of 16
    r16_matches = [(r32_winners[i], r32_winners[i+1])
                   for i in range(0, 16, 2)]
    r16_winners = play_knockout_round(r16_matches)
    for team in r16_winners:
        team_stages[team] = 'Quarter-final'
        # losers stay at 'Round of 16'
        
    #Quarter final
    qf_matches = [(r16_winners[i], r16_winners[i+1])
                  for i in range(0, 8, 2)]
    qf_winners = play_knockout_round(qf_matches)
    for team in qf_winners:
        team_stages[team] = 'Semi-final'
        # Team stay at 'Quarter final'

    # Semi finals
    sf_matches = [(qf_winners[i], qf_winners[i+1])
                  for i in range(0, 4, 2)]
    sf_winners = play_knockout_round(sf_matches)
    sf_losers  = [t for t in qf_winners if t not in sf_winners]

    # Semi-final losers go to Third Place
    for team in sf_losers:
        team_stages[team] = 'Third Place'
    # Semi-final winners advance to Final
    for team in sf_winners:
        team_stages[team] = 'Final'
    
    # Final
    final_winner = play_knockout_round([(sf_winners[0], sf_winners[1])])
    champion  = final_winner[0]
    runner_up = [t for t in sf_winners if t != champion][0]

    team_stages[champion]  = 'Winner'
    team_stages[runner_up] = 'Runner Up'

    return team_stages


# Profile new speed 
import time

start = time.time()
team_stages = run_tournament_fast(lambda_df)
elapsed = time.time() - start

print(f"Single tournament run: {elapsed:.3f} seconds")
print(f"Estimated 10,000 runs: {elapsed * 10000 / 60:.1f} minutes")

Single tournament run: 0.040 seconds
Estimated 10,000 runs: 6.6 minutes


## Monte Carlo Simulator — 10,000 Runs

### What it does
Runs the complete tournament simulation 10,000 times.
Each run produces a different result due to Bayesian λ sampling.
After all runs we count how often each team reached each stage.

### Why 10,000?
The law of large numbers — with enough runs the frequency
of each outcome converges to its true probability.

- 100 runs → noisy, Turkey might show 5% winner probability
- 1,000 runs → more stable but still some variance
- 10,000 runs → stable estimates, results won't change much
  if you run it again

### What gets tracked
For every team, every run:
### Converting to probabilities
After 10,000 runs:

In [21]:
#  Reset stage counts before rerunning 
from collections import defaultdict

stage_counts = defaultdict(lambda: defaultdict(int))
print("stage_counts reset — ready for fresh Monte Carlo run")

stage_counts reset — ready for fresh Monte Carlo run


In [22]:
# Monte Carlo simulator — 10,000 runs 
# Runs the full tournament 10,000 times and counts how often
# each team reaches each stage.
# Probabilities = counts / total runs

from collections import defaultdict
import time

N_SIMULATIONS = 10000

# Track stage counts for every team
stage_counts = defaultdict(lambda: defaultdict(int))

stages = ['Group Stage', 'Round of 32', 'Round of 16',
          'Quarter-final', 'Semi-final', 'Third Place',
          'Final', 'Runner Up', 'Winner']

print(f"Running {N_SIMULATIONS:,} simulations...")
start = time.time()

for sim in range(N_SIMULATIONS):
    # Progress update every 1000 runs
    if sim % 1000 == 0:
        elapsed = time.time() - start
        print(f"  Simulation {sim:,} / {N_SIMULATIONS:,}  ({elapsed:.0f}s elapsed)")

    team_stages = run_tournament_fast(lambda_df)

    for team, stage in team_stages.items():
        stage_counts[team][stage] += 1

elapsed = time.time() - start
print(f"\nDone — {N_SIMULATIONS:,} simulations in {elapsed:.0f} seconds")

Running 10,000 simulations...
  Simulation 0 / 10,000  (0s elapsed)
  Simulation 1,000 / 10,000  (36s elapsed)
  Simulation 2,000 / 10,000  (72s elapsed)
  Simulation 3,000 / 10,000  (109s elapsed)
  Simulation 4,000 / 10,000  (145s elapsed)
  Simulation 5,000 / 10,000  (181s elapsed)
  Simulation 6,000 / 10,000  (217s elapsed)
  Simulation 7,000 / 10,000  (253s elapsed)
  Simulation 8,000 / 10,000  (290s elapsed)
  Simulation 9,000 / 10,000  (326s elapsed)

Done — 10,000 simulations in 363 seconds


In [26]:
#  Debug — check what stages are actually being recorded 
# Look at one tournament run and print all team stages

team_stages = run_tournament_fast(lambda_df)

from collections import Counter
stage_counter = Counter(team_stages.values())
print("Stages recorded in one run:")
for stage, count in sorted(stage_counter.items()):
    print(f"  {stage:<20} {count} teams")

Stages recorded in one run:
  Group Stage          16 teams
  Quarter-final        4 teams
  Round of 16          8 teams
  Round of 32          16 teams
  Runner Up            1 teams
  Third Place          2 teams
  Winner               1 teams


In [27]:
#  Convert counts to probabilities 
# A team that reached the Final also reached the Semi-final
# We need cumulative counts not just final stage counts

stages_ordered = ['Round of 32', 'Round of 16', 'Quarter-final',
                  'Semi-final', 'Final', 'Winner']

# Map final stage to all stages reached
def stages_reached(final_stage):
    stage_map = {
        'Group Stage':    [],
        'Round of 32':    ['Round of 32'],
        'Round of 16':    ['Round of 32', 'Round of 16'],
        'Quarter-final':  ['Round of 32', 'Round of 16', 'Quarter-final'],
        'Third Place':    ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final'],
        'Semi-final':     ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final'],
        'Final':          ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final', 'Final'],
        'Runner Up':      ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final', 'Final'],
        'Winner':         ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final', 'Final'],
    }
    return stage_map.get(final_stage, [])

# Recount from stage_counts using cumulative logic
cumulative_counts = defaultdict(lambda: defaultdict(int))

for team in fixture_teams:
    for stage, count in stage_counts[team].items():
        for s in stages_reached(stage):
            cumulative_counts[team][s] += count

# Build results table
results = []
for team in fixture_teams:
    row = {'Team': team}
    row['Group Stage exit'] = round(stage_counts[team]['Group Stage'] / N_SIMULATIONS * 100, 1)
    for stage in stages_ordered:
        count = cumulative_counts[team][stage]
        row[stage] = round(count / N_SIMULATIONS * 100, 1)
    row['Winner'] = round(stage_counts[team]['Winner'] / N_SIMULATIONS * 100, 1)
    results.append(row)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Winner', ascending=False).reset_index(drop=True)
results_df.index += 1

print("WC26 Tournament Probabilities (%) — 10,000 simulations")
print("=" * 110)
print(results_df.to_string())

WC26 Tournament Probabilities (%) — 10,000 simulations
                    Team  Group Stage exit  Round of 32  Round of 16  Quarter-final  Semi-final  Final  Winner
1                  Spain               3.9         96.1         70.7           45.6        27.8   16.9    11.2
2              Argentina               6.7         93.3         67.9           41.3        28.1   18.4    10.8
3                 France              12.1         87.9         55.9           35.0        21.4   13.0     7.0
4                 Brazil              10.7         89.3         53.5           34.0        17.8   10.5     5.7
5                 Mexico               6.1         93.9         52.9           31.1        17.8    9.9     5.4
6                Ecuador              11.6         88.4         54.5           33.0        18.5    9.2     4.6
7                Germany              14.5         85.5         51.5           30.9        17.8    8.5     4.5
8               Portugal              14.0         86.0  

## Monte Carlo Simulation Results — 10,000 Runs

### Methodology
- 10,000 full tournament simulations completed in 376 seconds
- Each run uses Bayesian λ sampling — different result every time
- Probabilities = how often each team reached each stage / 10,000

### Tournament Winner Probabilities

**Top 5 favourites:**
| Team | Win % | Final % | Semi-final % |
|---|---|---|---|
| Spain | 11.2% | 16.9% | 27.8% |
| Argentina | 10.8% | 18.4% | 28.1% |
| France | 7.0% | 13.0% | 21.4% |
| Brazil | 5.7% | 10.5% | 17.8% |
| Mexico | 5.4% | 9.9% | 17.8% |

### Key Findings

**Spain and Argentina are co-favourites**
Essentially tied at 11.2% and 10.8% — reflecting their near-identical
Elo ratings and squad values going into the tournament.

**Mexico benefits from home advantage**
5.4% winner probability — significantly boosted by playing group
stage matches in front of home crowds in Mexico City and Guadalajara.

**Ecuador is the dark horse**
4.6% winner probability — higher than their FIFA rank suggests.
Strong Elo rating and squad value driving this.

**United States underperforms expectations**
Only 0.7% despite being a host nation. Home advantage helps
but the squad quality gap vs top teams is too large to overcome.

**Qatar near zero**
0.0% winner probability — correctly identified as the weakest
team in the tournament. 69% chance of group stage elimination.

### Group Stage Exit Rates
- Top teams (Spain, Argentina) — only 4-7% eliminated in groups
- Host nations (Canada 6.5%, Mexico 6.1%) — home advantage keeping
  them safe in the group stage
- Weakest teams (Ghana, Curaçao, Qatar) — 70%+ eliminated in groups

### Caveat
These are pre-tournament probabilities based on historical data,
Elo ratings, and squad values as of June 2026. They do not account
for injuries, team news, or any matches already played in WC26.

In [28]:
# Save simulation results 

results_df.to_csv(processed_dir / 'simulation_results.csv', index=False)

# Also save raw stage counts for reference
stage_counts_df = pd.DataFrame([
    {'team': team, 'stage': stage, 'count': count}
    for team, stages in stage_counts.items()
    for stage, count in stages.items()
])
stage_counts_df.to_csv(processed_dir / 'stage_counts_raw.csv', index=False)

print("Saved:")
print(f"  {processed_dir}/simulation_results.csv")
print(f"  {processed_dir}/stage_counts_raw.csv")

print(f"\nSimulation summary:")
print(f"  Total runs:      {N_SIMULATIONS:,}")
print(f"  Runtime:         376 seconds")
print(f"  Tournament winner: Spain {results_df[results_df['Team']=='Spain']['Winner'].values[0]}%")
print(f"  Biggest surprise: {results_df.iloc[-1]['Team']} {results_df.iloc[-1]['Winner']}% winner probability")

Saved:
  /home/ubuntu/wc2026/data/processed/simulation_results.csv
  /home/ubuntu/wc2026/data/processed/stage_counts_raw.csv

Simulation summary:
  Total runs:      10,000
  Runtime:         376 seconds
  Tournament winner: Spain 11.2%
  Biggest surprise: Qatar 0.0% winner probability


In [29]:
#  Save everything needed for dashboard 

import pickle

# Save results_df — main probability table
results_df.to_csv(processed_dir / 'simulation_results.csv', index=False)

# Save team features — for team profiles on dashboard
team_features.to_csv(processed_dir / 'team_features_2026.csv')

# Save lambda_df — for showing expected goals per fixture
lambda_df.to_csv(processed_dir / 'fixture_lambdas.csv', index=False)

# Save fixtures — for group and bracket display
fixtures.to_csv(processed_dir / 'fixtures_2026_clean.csv', index=False)

# Save group stage structure — group names and teams
group_info = {}
for group in sorted(fixtures[fixtures['stage']=='Group']['group'].unique()):
    group_fix = fixtures[fixtures['group']==group]
    teams = pd.concat([
        group_fix['home_team'],
        group_fix['away_team']
    ]).unique().tolist()
    group_info[group] = teams

with open(processed_dir / 'group_info.pkl', 'wb') as f:
    pickle.dump(group_info, f)

print("Saved for dashboard:")
print(f"  simulation_results.csv   — team probabilities at every stage")
print(f"  team_features_2026.csv   — Elo, FIFA rank, squad value per team")
print(f"  fixture_lambdas.csv      — expected goals per fixture")
print(f"  fixtures_2026_clean.csv  — full fixture list with venues")
print(f"  group_info.pkl           — group structure")

Saved for dashboard:
  simulation_results.csv   — team probabilities at every stage
  team_features_2026.csv   — Elo, FIFA rank, squad value per team
  fixture_lambdas.csv      — expected goals per fixture
  fixtures_2026_clean.csv  — full fixture list with venues
  group_info.pkl           — group structure
